In [1]:
import os
import pandas as pd

In [2]:
data_folder = "data"

files = [f for f in os.listdir(data_folder) if f.endswith(".csv")]
print("Files found:", files)

df_list = []

for file in files:
    temp = pd.read_csv(os.path.join(data_folder, file))
    temp["Season"] = file.replace(".csv", "")
    df_list.append(temp)

df = pd.concat(df_list, ignore_index=True)

print("Shape:", df.shape)
df.head()

Files found: ['SP1 (7).csv', 'SP1 (6).csv', 'SP1 (1).csv', 'SP1.csv', 'SP1 (3).csv', 'SP1 (2).csv', 'SP1 (9).csv', 'SP1 (5).csv', 'SP1 (4).csv', 'SP1 (8).csv']
Shape: (3700, 180)


,Div,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,HTR,...,BMGMCA,BVCH,BVCD,BVCA,CLCH,CLCD,CLCA,LBCH,LBCD,LBCA
0,SP1,18/08/17,Leganes,Alaves,1,0,H,1,0,H,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,SP1,18/08/17,Valencia,Las Palmas,1,0,H,1,0,H,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,SP1,19/08/17,Celta,Sociedad,2,3,A,1,1,D,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,SP1,19/08/17,Girona,Ath Madrid,2,2,D,2,0,H,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,SP1,19/08/17,Sevilla,Espanol,1,1,D,1,1,D,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
missing = df.isnull().sum()
print(missing[missing > 0].sort_values(ascending=False))

LBCA    3482
CLCH    3482
LBCD    3482
LBCH    3482
CLCA    3482
        ... 
PSD       95
PSH       95
PSCH      93
PSCD      93
PSCA      93
Length: 154, dtype: int64


In [4]:
threshold = len(df) * 0.5
df_clean = df.dropna(axis=1, thresh=threshold)

print("Original shape:", df.shape)
print("After dropping sparse columns:", df_clean.shape)

Original shape: (3700, 180)
After dropping sparse columns: (3700, 103)


In [5]:
columns_to_keep = [
    'Date', 'HomeTeam', 'AwayTeam',
    'FTHG', 'FTAG', 'FTR',
    'HTHG', 'HTAG', 'HTR',
    'HS', 'AS', 'HST', 'AST',
    'HF', 'AF', 'HC', 'AC',
    'HY', 'AY', 'HR', 'AR',
    'Season'
]

df_clean = df_clean[columns_to_keep]
print(df_clean.shape)
df_clean.head()

(3700, 22)


,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,HTR,HS,...,AST,HF,AF,HC,AC,HY,AY,HR,AR,Season
0,18/08/17,Leganes,Alaves,1,0,H,1,0,H,16,...,3,14,18,4,2,0,1,0,0,SP1 (7)
1,18/08/17,Valencia,Las Palmas,1,0,H,1,0,H,22,...,4,25,13,5,2,3,3,0,1,SP1 (7)
2,19/08/17,Celta,Sociedad,2,3,A,1,1,D,16,...,6,12,11,5,4,3,1,0,0,SP1 (7)
3,19/08/17,Girona,Ath Madrid,2,2,D,2,0,H,13,...,3,15,15,6,0,2,4,0,1,SP1 (7)
4,19/08/17,Sevilla,Espanol,1,1,D,1,1,D,9,...,6,14,12,7,3,2,4,1,0,SP1 (7)


In [6]:
raw_dates = df_clean['Date'].astype(str).str.strip()

date_long = pd.to_datetime(raw_dates, format='%d/%m/%Y', errors='coerce')
date_short = pd.to_datetime(raw_dates, format='%d/%m/%y', errors='coerce')

df_clean['Date'] = date_long.fillna(date_short)

print(df_clean[['Date']].head(10))
print("Missing dates:", df_clean['Date'].isnull().sum())

        Date
0 2017-08-18
1 2017-08-18
2 2017-08-19
3 2017-08-19
4 2017-08-19
5 2017-08-20
6 2017-08-20
7 2017-08-20
8 2017-08-21
9 2017-08-21
Missing dates: 0


In [ ]:
df_clean = df_clean.dropna(subset=['Date'])
print("After dropping missing dates:", df_clean.shape) # drop rows where there is no Date

After dropping missing dates: (3700, 22)


In [ ]:
df_clean = df_clean.sort_values('Date').reset_index(drop=True)
df_clean.head() # Sorting from oldest to newest (checking only first 5)

,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,HTR,HS,...,AST,HF,AF,HC,AC,HY,AY,HR,AR,Season
0,2015-08-21,Malaga,Sevilla,0,0,D,0,0,D,25,...,2,19,11,7,2,3,3,0,1,SP1 (9)
1,2015-08-22,La Coruna,Sociedad,0,0,D,0,0,D,15,...,2,16,10,5,4,3,2,0,0,SP1 (9)
2,2015-08-22,Vallecano,Valencia,0,0,D,0,0,D,8,...,4,19,11,3,3,3,1,0,0,SP1 (9)
3,2015-08-22,Ath Madrid,Las Palmas,1,0,H,1,0,H,14,...,1,11,16,4,4,2,0,0,0,SP1 (9)
4,2015-08-22,Espanol,Getafe,1,0,H,1,0,H,4,...,3,19,14,5,6,2,3,0,1,SP1 (9)


In [ ]:
print(df_clean[['Date']].head(5))
print(df_clean[['Date']].tail(5)) #check if in order

        Date
0 2015-08-21
1 2015-08-22
2 2015-08-22
3 2015-08-22
4 2015-08-22
           Date
3695 2026-03-15
3696 2026-03-15
3697 2026-03-15
3698 2026-03-15
3699 2026-03-16


In [ ]:
print(df_clean.isnull().sum()) #check to see if data is cleaned 

Date        0
HomeTeam    0
AwayTeam    0
FTHG        0
FTAG        0
FTR         0
HTHG        0
HTAG        0
HTR         0
HS          0
AS          0
HST         0
AST         0
HF          0
AF          0
HC          0
AC          0
HY          0
AY          0
HR          0
AR          0
Season      0
dtype: int64


In [15]:
df_clean.head(50)

,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,HTR,HS,...,AST,HF,AF,HC,AC,HY,AY,HR,AR,Season
0,2015-08-21,Malaga,Sevilla,0,0,D,0,0,D,25,...,2,19,11,7,2,3,3,0,1,SP1 (9)
1,2015-08-22,La Coruna,Sociedad,0,0,D,0,0,D,15,...,2,16,10,5,4,3,2,0,0,SP1 (9)
2,2015-08-22,Vallecano,Valencia,0,0,D,0,0,D,8,...,4,19,11,3,3,3,1,0,0,SP1 (9)
3,2015-08-22,Ath Madrid,Las Palmas,1,0,H,1,0,H,14,...,1,11,16,4,4,2,0,0,0,SP1 (9)
4,2015-08-22,Espanol,Getafe,1,0,H,1,0,H,4,...,3,19,14,5,6,2,3,0,1,SP1 (9)
5,2015-08-23,Sp Gijon,Real Madrid,0,0,D,0,0,D,6,...,8,20,4,2,5,4,0,0,0,SP1 (9)
6,2015-08-23,Ath Bilbao,Barcelona,0,1,A,0,0,D,8,...,4,16,11,2,4,3,3,0,0,SP1 (9)
7,2015-08-23,Betis,Villarreal,1,1,D,0,1,A,22,...,4,11,26,7,5,3,1,0,1,SP1 (9)
8,2015-08-23,Levante,Celta,1,2,A,0,1,A,11,...,6,12,17,4,6,1,3,1,0,SP1 (9)
9,2015-08-24,Granada,Eibar,1,3,A,0,2,A,12,...,4,5,23,6,5,1,6,1,0,SP1 (9)
